# SETUP & IMPORTS


In [ ]:
# Install required libraries (run once in Colab)
!pip install -U langgraph langchain_groq joblib pandas numpy
!pip install langchain langchain-core langchain_community
!pip install faiss-cpu sentence-transformers groq PyPDF2
!pip install langchain-huggingface sentence-transformers

# Core libraries
import joblib
import pandas as pd
import numpy as np

# Typing & utilities
from typing import TypedDict, List, Annotated, Union
import operator

# LangChain messages
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# LOAD TRAINED MODELS



In [ ]:
# Regression, Classification, Clustering models
reg_model = joblib.load('linear_model.pkl')
clf_model = joblib.load('logistic_model.pkl')
cluster_model = joblib.load('kmeans_model.pkl')

# Scalers
scaler_reg = joblib.load('scaler_reg.pkl')
scaler_clf = joblib.load('scaler_clf.pkl')
scaler_cluster = joblib.load('scaler_cluster.pkl')


# MAIN ML PIPELINE FUNCTION


def run_ml_pipeline(user_data: dict):
    """
    Takes user input (student data) and returns:
    - Predicted exam score
    - Pass/Fail status
    - Performance category (cluster)
    """

    # ------------------------------
    # 1. DEFAULT INPUT HANDLING
    # ------------------------------
    data = {
        # Numerical defaults
        'MathScore': user_data.get('math', 67.0),
        'ReadingScore': user_data.get('reading', 70.0),
        'WklyStudyHours': user_data.get('study_hours', 7.5),
        'NrSiblings': user_data.get('siblings', 2.0),

        # Categorical (raw)
        'ParentEduc': user_data.get('parent_educ', 2),
        'PracticeSport': user_data.get('sport', 1),

        # One-hot encoded features
        'TestPrep_none': 1 if user_data.get('test_prep', 'none') == 'none' else 0,
        'LunchType_standard': 1 if user_data.get('lunch', 'standard') == 'standard' else 0,
        'Gender_male': 1 if user_data.get('gender', 'female') == 'male' else 0,
        'IsFirstChild_yes': 1 if user_data.get('is_first_child', 'yes') == 'yes' else 0,
        'TransportMeans_school_bus': 1 if user_data.get('transport', 'school_bus') == 'school_bus' else 0
    }

    # ------------------------------
    # 2. FEATURE PREPARATION
    # ------------------------------
    feat_cols = [
        'MathScore', 'ReadingScore', 'WklyStudyHours', 'ParentEduc',
        'TestPrep_none', 'LunchType_standard', 'PracticeSport',
        'NrSiblings', 'Gender_male', 'IsFirstChild_yes',
        'TransportMeans_school_bus'
    ]

    X_input = pd.DataFrame([data])[feat_cols]

    # ------------------------------
    # 3. REGRESSION (Score Prediction)
    # ------------------------------
    X_reg_scaled = scaler_reg.transform(X_input)
    pred_exam_score = reg_model.predict(X_reg_scaled)[0]

    # ------------------------------
    # 4. CLASSIFICATION (Pass/Fail)
    # ------------------------------
    X_clf_scaled = scaler_clf.transform(X_input)
    pass_fail = clf_model.predict(X_clf_scaled)[0]

    # ------------------------------
    # 5. CLUSTERING (Performance Group)
    # ------------------------------
    cluster_data = {
        'ExamScore': pred_exam_score,
        'WklyStudyHours': data['WklyStudyHours'],
        'ParentEduc': data['ParentEduc'],
        'LunchType_standard': data['LunchType_standard'],
        'TestPrep_none': data['TestPrep_none'],
        'PracticeSport': data['PracticeSport']
    }

    cluster_cols = [
        'ExamScore', 'WklyStudyHours', 'ParentEduc',
        'LunchType_standard', 'TestPrep_none', 'PracticeSport'
    ]

    X_cluster_input = pd.DataFrame([cluster_data])[cluster_cols]
    X_cluster_scaled = scaler_cluster.transform(X_cluster_input)
    cluster_id = cluster_model.predict(X_cluster_scaled)[0]

    # ------------------------------
    # 6. CLUSTER LABEL MAPPING
    # ------------------------------
    category_map = {
        0: "At-Risk",
        1: "Average",
        2: "High-Performer"
    }

    # ------------------------------
    # 7. FINAL OUTPUT
    # ------------------------------
    return {
        "predicted_score": round(pred_exam_score, 2),
        "status": pass_fail,
        "category": category_map.get(cluster_id, "Unknown")
    }

# GRAPH STATE (Agent Memory)



In [1]:
from typing import Annotated, List, TypedDict
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    """
    Central state object passed between nodes in the graph.
    Acts like a shared memory ("clipboard") for the agent.
    """

    # Conversation history
    messages: Annotated[List[BaseMessage], operator.add]

    # Input + ML results
    student_data: dict
    ml_results: dict

    # Retrieval / RAG
    retrieved_docs: List[str]

    # Quiz system
    quiz_questions: List[dict]
    current_q_idx: int
    quiz_score: int
    quiz_active: bool
    awaiting_answer: bool

    # Study planning
    study_plan: str
    planner_notes: str
    plan: List[str]
    current_step_index: int

    # Flow control
    next_node: str


# LLM SETUP (Groq)


In [2]:
from langchain_groq import ChatGroq
import os

# Set your API key properly
os.environ["GROQ_API_KEY"] = "your_groq_api_key"

# Initialize LLM
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0
)

ModuleNotFoundError: No module named 'langchain_groq'


# DATA EXTRACTION LOGIC



In [ ]:
from pydantic import BaseModel, Field
from typing import Optional
from langchain_core.messages import AIMessage, HumanMessage
import json


# ------------------------------
# 1. STRUCTURED SCHEMA
# ------------------------------
class StudentDataSchema(BaseModel):
    """
    Defines the structure for extracting student data
    from natural language input.
    """

    math: Optional[int] = Field(None, description="Math score (0-100)")
    reading: Optional[int] = Field(None, description="Reading score (0-100)")
    study_hours: Optional[float] = Field(None, description="Weekly study hours")

    parent_educ: Optional[int] = Field(
        None,
        description="Education level: high_school=1, some_college=2, associates=3, bachelors=4, masters=5"
    )

    test_prep: Optional[str] = Field(
        None,
        description="Test preparation: 'none' or 'completed'"
    )

    lunch: Optional[str] = Field(
        None,
        description="Lunch type: 'standard' or 'free/reduced'"
    )

    sport: Optional[int] = Field(
        None,
        description="Sports participation: never=0, sometimes=1, regularly=2"
    )

    gender: Optional[str] = Field(
        None,
        description="'male' or 'female'"
    )


# ------------------------------
# 2. EXTRACTION FUNCTION
# ------------------------------
def extraction_logic(user_message: str):
    """
    Extracts structured student data from user input using LLM.
    Returns only valid, non-null fields.
    """

    # Enforce schema-based structured output
    structured_llm = llm.with_structured_output(StudentDataSchema)

    prompt = f"""
    You are a precise data extraction system.

    Extract only explicitly stated or clearly implied values.
    Do NOT guess missing information.

    USER INPUT:
    "{user_message}"
    """

    try:
        extracted_obj = structured_llm.invoke(prompt)

        # Convert to dictionary
        raw_data = extracted_obj.dict()

        # Remove None values (prevents overwriting existing state)
        clean_data = {
            key: value
            for key, value in raw_data.items()
            if value is not None
        }

        return clean_data

    except Exception as e:
        print(f"[Extraction Error]: {e}")
        return {}


# ANALYSER NODE


In [ ]:
def analyser_node(state: AgentState):
    """
    Extracts student data from user input,
    updates state, runs ML pipeline,
    and generates a response.
    """

    print("--- NODE: ANALYSER ---")

    # ------------------------------
    # 1. GET LATEST USER INPUT
    # ------------------------------
    last_human_msg = [
        msg.content
        for msg in state["messages"]
        if isinstance(msg, HumanMessage)
    ][-1]

    # ------------------------------
    # 2. EXTRACT NEW DATA
    # ------------------------------
    newly_extracted_data = extraction_logic(last_human_msg)

    # ------------------------------
    # 3. MERGE WITH EXISTING STATE
    # ------------------------------
    current_student_data = state.get("student_data", {}).copy()
    current_student_data.update(newly_extracted_data)

    # ------------------------------
    # 4. RUN ML PIPELINE
    # ------------------------------
    ml_results = run_ml_pipeline(current_student_data)

    # ------------------------------
    # 5. GENERATE RESPONSE
    # ------------------------------
    if newly_extracted_data:
        captured_fields = ", ".join(newly_extracted_data.keys())

        analysis_msg = (
            f"I've updated your profile with: {captured_fields}. "
            f"Your predicted exam score is {ml_results['predicted_score']} "
            f"(Status: {ml_results['status']}). "
            f"You are currently in the '{ml_results['category']}' category."
        )
    else:
        analysis_msg = (
            "I couldn't extract new academic details from your message. "
            "To improve accuracy, please share things like your math or reading scores."
        )

    # ------------------------------
    # 6. RETURN UPDATED STATE
    # ------------------------------
    return {
        "student_data": current_student_data,
        "ml_results": ml_results,
        "messages": [AIMessage(content=analysis_msg)],
        "next_node": "supervisor"
    }


# EXECUTION PLAN (Router Schema)


In [ ]:
from typing import List, Literal
from pydantic import BaseModel, Field


class ExecutionPlan(BaseModel):
    """
    Defines the sequence of nodes the agent should execute.
    Generated by the planning LLM ("Architect").
    """

    steps: List[Literal[
        "analyser",
        "retriever",
        "planner",
        "quizzer",
        "end"
    ]] = Field(
        description="Ordered list of nodes to execute. Use [] for off-topic queries."
    )

    reasoning: str = Field(
        description="Brief justification for the selected execution steps."
    )


# MASTER NODE (ARCHITECT)


In [ ]:
from langchain_core.messages import HumanMessage

def master_node(state: AgentState):
    """
    Decides what the agent should do next by:
    - Creating an execution plan (if needed)
    - Executing steps sequentially
    """

    print("--- NODE: MASTER (ARCHITECT) ---")

    # ------------------------------
    # 1. QUIZ BYPASS
    # ------------------------------
    if state.get("quiz_active", False):
        return {"next_node": "quizzer"}

    plan = state.get("plan", [])
    step_idx = state.get("current_step_index", 0)

    # ------------------------------
    # 2. CREATE NEW PLAN (IF NEEDED)
    # ------------------------------
    if not plan or step_idx >= len(plan):

        user_msg = [
            m.content for m in state["messages"]
            if isinstance(m, HumanMessage)
        ][-1]

        user_msg_lower = user_msg.lower()

        architect_llm = llm.with_structured_output(ExecutionPlan)

        system_prompt = f"""
You are an Agentic System Architect.

Create a minimal, correct, and efficient execution plan.

--- NODE RULES ---
- analyser → ONLY if performance/data needed
- retriever → ONLY if concept/explanation needed
- planner → ONLY if study plan requested
- quizzer → ONLY if quiz requested

--- STATE-AWARE RULES ---
- Skip analyser if ml_results already exist
- Skip retriever if retrieved_docs already exist

--- DEPENDENCIES ---
- analyser before planner
- retriever before planner (if topic needed)

--- CONSTRAINTS ---
- No duplicate steps
- Always end with "end"
- Off-topic → ["end"]

USER QUERY:
{user_msg}
"""

        new_plan_obj = architect_llm.invoke(system_prompt)
        new_plan = new_plan_obj.steps

        # ------------------------------
        # 3. GUARDRAILS
        # ------------------------------

        valid_nodes = {"analyser", "retriever", "planner", "quizzer", "end"}

        # Remove invalid nodes
        new_plan = [step for step in new_plan if step in valid_nodes]

        # Remove duplicates (preserve order)
        new_plan = list(dict.fromkeys(new_plan))

        # Ensure 'end'
        if "end" not in new_plan:
            new_plan.append("end")

        # Dependency correction
        if "planner" in new_plan and "retriever" not in new_plan:
            if any(word in user_msg_lower for word in ["explain", "concept", "topic"]):
                new_plan.insert(0, "retriever")

        # State-aware cleanup
        if "analyser" in new_plan and state.get("ml_results"):
            new_plan.remove("analyser")

        if "retriever" in new_plan and state.get("retrieved_docs"):
            new_plan.remove("retriever")

        # Fallback safety
        if not new_plan:
            new_plan = ["end"]

        print(f"Plan → {new_plan}")
        print(f"Reason → {new_plan_obj.reasoning}")

        return {
            "plan": new_plan,
            "current_step_index": 1,
            "next_node": new_plan[0]
        }

    # ------------------------------
    # 4. EXECUTE NEXT STEP
    # ------------------------------
    next_node = plan[step_idx]

    print(f"Executing Step {step_idx + 1}/{len(plan)} → {next_node}")

    return {
        "current_step_index": step_idx + 1,
        "next_node": next_node
    }


# STUDY PLANNER NODE


In [ ]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.messages import AIMessage, HumanMessage


# ------------------------------
# 1. PLAN SCHEMA
# ------------------------------
class StudyDay(BaseModel):
    day: str = Field(description="e.g., Day 1")
    topic: str = Field(description="Focus area for the day")
    activities: List[str] = Field(description="Specific tasks or exercises")
    estimated_time: str = Field(description="e.g., 2 hours")


class WeeklyPlan(BaseModel):
    title: str
    days: List[StudyDay]
    summary: str


# ------------------------------
# 2. PLANNER NODE
# ------------------------------
def planner_node(state: AgentState):
    """
    Generates a structured 7-day study plan using:
    - User intent
    - Retrieved learning resources (RAG)
    - Optional ML performance context
    """

    print("--- NODE: PLANNER ---")

    # ------------------------------
    # CONTEXT COLLECTION
    # ------------------------------
    results = state.get("ml_results", {})
    category = results.get("category", "Unknown")
    score = results.get("predicted_score", "N/A")

    last_user_msg = [
        m.content for m in state["messages"]
        if isinstance(m, HumanMessage)
    ][-1]

    external_knowledge = "\n".join(state.get("retrieved_docs", []))

    planner_llm = llm.with_structured_output(WeeklyPlan)

    # ------------------------------
    # PROMPT
    # ------------------------------
    prompt = f"""
You are an expert Study Coach.

STUDENT GOAL:
"{last_user_msg}"

LEARNING RESOURCES:
"{external_knowledge}"

ACADEMIC CONTEXT:
- Category: {category}
- Score: {score}

TASK:
Create a structured 7-day study plan.

RULES:
1. If data is missing → rely on goal + resources
2. Use resources to define activities
3. Adapt difficulty based on category:
   - At-Risk → slower pace, 1 concept/day
   - High-Performer → multi-topic + challenge on Day 7
   - Unknown → balanced plan (no mention of missing data)
"""

    plan = planner_llm.invoke(prompt)

    # ------------------------------
    # FORMAT OUTPUT (MARKDOWN)
    # ------------------------------
    markdown_plan = f"### 🗓️ {plan.title}\n"

    if category != "Unknown":
        markdown_plan += f"*Optimized for {category} performance*\n\n"
    else:
        markdown_plan += "*Standard balanced plan*\n\n"

    for d in plan.days:
        markdown_plan += f"**{d.day}: {d.topic}**\n"
        markdown_plan += "- " + "\n- ".join(d.activities)
        markdown_plan += f"\n*Duration: {d.estimated_time}*\n\n"

    # FIXED BUG HERE (important)
    if plan.summary.strip():
        markdown_plan += f"---\n**How to Approach This Plan:** {plan.summary}"

    return {
        "study_plan": markdown_plan,
        "messages": [AIMessage(content=markdown_plan)],
        "next_node": "supervisor"
    }


# RETRIEVER NODE (CUSTOM RAG)


In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from langchain_core.messages import AIMessage


# ------------------------------
# 1. KNOWLEDGE BASE
# ------------------------------
KNOWLEDGE_BASE = [

    # ── ALGEBRA ──
    "Algebra - Linear Equations: A linear equation is an equation of the form ax + b = 0. To solve it, isolate the variable by performing inverse operations (addition, subtraction, multiplication, division) on both sides equally. Example: 2x + 4 = 0 → 2x = -4 → x = -2. Key idea: maintain balance on both sides.",

    "Algebra - Quadratic Equations: A quadratic equation has the form ax² + bx + c = 0. It can be solved using factorization, completing the square, or the quadratic formula: x = (-b ± √(b² - 4ac)) / (2a). The discriminant (b² - 4ac) determines the nature of roots (real, equal, or complex).",

    "Algebra - Inequalities: Inequalities are similar to equations but involve signs like >, <, ≥, ≤. When multiplying or dividing both sides by a negative number, the inequality sign must be reversed.",

    # ── ARITHMETIC ──
    "Arithmetic - Percentages: Percentage represents parts per hundred. Formula: percentage = (part/whole) × 100. Used in profit-loss, discounts, and data interpretation problems.",

    "Arithmetic - Ratios & Proportions: A ratio compares two quantities, while proportion states equality of two ratios. Example: a/b = c/d. Used in scaling, mixtures, and real-life comparisons.",

    # ── GEOMETRY ──
    "Geometry - Triangles: The sum of interior angles of a triangle is 180°. In a right triangle, Pythagoras theorem applies: a² + b² = c². Used to find missing sides.",

    "Geometry - Circles: Key elements include radius, diameter, circumference (2πr), and area (πr²). Understanding relationships between these is essential for solving problems.",

    # ── TRIGONOMETRY ──
    "Trigonometry - Basics: Trigonometric ratios (sin, cos, tan) relate angles to sides of a right triangle. sin = opposite/hypotenuse, cos = adjacent/hypotenuse, tan = opposite/adjacent.",

    # ── STATISTICS & PROBABILITY ──
    "Statistics - Central Tendency: Mean is average, median is middle value, mode is most frequent value. Used to summarize datasets.",

    "Probability: Probability measures likelihood of an event. Formula: favorable outcomes / total outcomes. Value lies between 0 and 1.",

    # ── FUNCTIONS & CALCULUS ──
    "Functions: A function maps an input to exactly one output. Graphs help visualize relationships between variables.",

    "Calculus - Derivatives: A derivative represents rate of change (slope of a curve). Example: speed is derivative of position.",

    "Calculus - Integrals: Integrals represent accumulation or area under a curve. They are the reverse of derivatives.",

    # ── READING & COMPREHENSION ──
    "Reading Comprehension: Focus on identifying main ideas, supporting details, tone, and inference. Always connect ideas across paragraphs.",

    "Inference Skills: Inference means understanding what is implied, not directly stated. Requires combining clues from the text.",

    # ── WRITING ──
    "Writing - Structure: Use PEEL method — Point, Evidence, Explanation, Link — to construct clear and logical paragraphs.",

    # ── PROBLEM SOLVING STRATEGY ──
    "Problem Solving: First understand the problem, identify knowns and unknowns, choose a method, solve step-by-step, and verify the answer.",

    "Error Analysis: Reviewing mistakes helps identify conceptual gaps and prevents repeating errors.",

    # ── STUDY SCIENCE ──
    "Active Recall: Testing yourself without notes strengthens memory more effectively than passive review.",

    "Spaced Repetition: Revisiting topics at intervals improves long-term retention.",

    # ── RESOURCES (IMPORTANT FOR RAG RESPONSES) ──
    "For Mathematics learning, use  (khanacademy.org) for structured courses from basics to advanced.",

    "For conceptual understanding,  provides visual explanations of math topics.",

    "For advanced topics,  offers free university-level lectures.",

    "For structured courses,  allows auditing courses for free.",

    # ── META LEARNING ──
    "If you cannot explain a concept simply, you have not fully understood it.",

    "Learning builds layer by layer; weak fundamentals lead to difficulty in advanced topics.",
]


# ------------------------------
# 2. EMBEDDING + INDEX SETUP
# ------------------------------
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(KNOWLEDGE_BASE)

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

# Map index → text
doc_store = {i: text for i, text in enumerate(KNOWLEDGE_BASE)}


# ------------------------------
# 3. RETRIEVER NODE
# ------------------------------
def retriever_node(state: AgentState):
    """
    Retrieves relevant knowledge using FAISS
    and generates a grounded response using LLM.
    """

    print("--- NODE: RETRIEVER (RAG) ---")

    user_query = state["messages"][-1].content
    category = state.get("ml_results", {}).get("category", "General")

    # ------------------------------
    # VECTOR SEARCH
    # ------------------------------
    query_vector = embedding_model.encode([user_query])

    k = 2
    distances, indices = index.search(np.array(query_vector), k)

    docs = [doc_store[i] for i in indices[0]]
    context = "\n".join(docs)

    # ------------------------------
    # LLM RESPONSE (GROUNDED)
    # ------------------------------
    prompt = f"""
You are an Academic Assistant.

User Category: {category}
User Question: {user_query}

Retrieved Knowledge:
{context}

RULES:
- Answer ONLY using retrieved knowledge
- Adapt explanation to the student's level
- If insufficient information, say "I don't know"
"""

    response = llm.invoke(prompt)

    return {
        "retrieved_docs": docs,
        "messages": [AIMessage(content=response.content)],
        "next_node": "supervisor"
    }


# QUIZZER NODE


In [ ]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.messages import AIMessage, HumanMessage


# ------------------------------
# 1. QUIZ SCHEMA
# ------------------------------
class Question(BaseModel):
    question: str
    options: List[str]
    correct_answer: str
    explanation: str


class QuizSet(BaseModel):
    topic: str
    questions: List[Question]


# ------------------------------
# 2. QUIZZER NODE
# ------------------------------
def quizzer_node(state: AgentState):
    """
    Handles:
    - Quiz generation
    - Answer evaluation
    - Progress tracking
    """

    print("--- NODE: QUIZZER ---")

    # ------------------------------
    # WAIT STATE (avoid double-trigger)
    # ------------------------------
    if state.get("awaiting_answer", False) and isinstance(state["messages"][-1], AIMessage):
        return {"next_node": "supervisor"}

    # ------------------------------
    # EXTRACT USER MESSAGE
    # ------------------------------
    last_user_msg = None
    for m in reversed(state["messages"]):
        if isinstance(m, HumanMessage):
            last_user_msg = m.content
            break

    if not last_user_msg:
        return {"next_node": "supervisor"}

    # ------------------------------
    # STATE VARIABLES
    # ------------------------------
    quiz_active = state.get("quiz_active", False)
    questions = state.get("quiz_questions", [])
    current_idx = state.get("current_q_idx", 0)
    score = state.get("quiz_score", 0)

    # ------------------------------
    # PHASE A: INITIALIZE QUIZ
    # ------------------------------
    if not quiz_active or not questions:
        print("Initializing Quiz...")

        quiz_llm = llm.with_structured_output(QuizSet)

        prompt = f"""
Generate a 3-question MCQ quiz on: {last_user_msg}
Ensure difficulty matches student level.
"""

        generated_quiz = quiz_llm.invoke(prompt)

        first_q = generated_quiz.questions[0]

        return {
            "quiz_questions": [q.model_dump() for q in generated_quiz.questions],
            "current_q_idx": 0,
            "quiz_score": 0,
            "quiz_active": True,
            "awaiting_answer": True,
            "messages": [AIMessage(content=
                f"Starting Quiz!\n\n"
                f"Q1: {first_q.question}\n"
                f"Options: {', '.join(first_q.options)}\n"
                f"👉 Reply with option text or A/B/C/D"
            )],
            "next_node": "supervisor"
        }

    # ------------------------------
    # PHASE B: EVALUATE ANSWER
    # ------------------------------
    current_q = questions[current_idx]

    user_ans = last_user_msg.strip().lower()
    correct_ans = current_q["correct_answer"].strip().lower()

    # Map A/B/C/D → actual option
    option_map = {
        chr(65 + i).lower(): opt.lower()
        for i, opt in enumerate(current_q["options"])
    }

    if user_ans in option_map:
        user_ans = option_map[user_ans]

    is_correct = user_ans == correct_ans
    new_score = score + 1 if is_correct else score

    feedback = "✅ Correct!" if is_correct else f"❌ Incorrect. Correct answer: {current_q['correct_answer']}"
    feedback += f"\nExplanation: {current_q['explanation']}"

    # ------------------------------
    # PHASE C: NEXT OR FINISH
    # ------------------------------
    next_idx = current_idx + 1

    if next_idx < len(questions):
        next_q = questions[next_idx]

        response_text = (
            f"{feedback}\n\n"
            f"--- NEXT QUESTION ---\n"
            f"Q{next_idx+1}: {next_q['question']}\n"
            f"Options: {', '.join(next_q['options'])}\n"
            f"👉 Reply with option text or A/B/C/D"
        )

        return {
            "current_q_idx": next_idx,
            "quiz_score": new_score,
            "awaiting_answer": True,
            "messages": [AIMessage(content=response_text)],
            "next_node": "supervisor"
        }

    # ------------------------------
    # QUIZ COMPLETE
    # ------------------------------
    final_msg = (
        f"{feedback}\n\n"
        f"🏆 Quiz Finished!\n"
        f"Final Score: {new_score}/{len(questions)}"
    )

    return {
        "quiz_active": False,
        "awaiting_answer": False,
        "quiz_score": new_score,
        "messages": [AIMessage(content=final_msg)],
        "next_node": "supervisor"
    }


# FINAL RESPONSE NODE


In [ ]:
from langchain_core.messages import AIMessage

def end_node(state: AgentState):
    """
    Generates the final user-facing response by:
    - Detecting intent
    - Selecting relevant context
    - Structuring output cleanly
    """

    print("--- NODE: END/RESPONSE ---")

    last_message = state["messages"][-1].content

    # ------------------------------
    # RESPONSE GENERATION PROMPT
    # ------------------------------
    prompt = f"""
You are an AI Study Coach generating the FINAL response.

USER MESSAGE:
"{last_message}"

AVAILABLE CONTEXT:
- Student Data: {state.get("student_data")}
- ML Analysis: {state.get("ml_results")}
- Retrieved Knowledge: {state.get("retrieved_docs")}
- Study Plan: {state.get("study_plan")}
- Planner Notes: {state.get("planner_notes")}
- Quiz Active: {state.get("quiz_active")}

--- CORE PRINCIPLE ---
Select ONLY relevant information. Do NOT merge everything blindly.

--- STEP 1: DETECT INTENTS ---
Possible intents:
- GREETING
- CLOSING
- STUDY_PLAN
- CONCEPT_EXPLANATION
- PERFORMANCE_INSIGHT
- QUIZ
- OFF_TOPIC

--- STEP 2: PRIORITY ---
1. QUIZ
2. STUDY_PLAN
3. PERFORMANCE_INSIGHT
4. CONCEPT_EXPLANATION
5. GREETING / CLOSING
6. OFF_TOPIC

--- STEP 3: CONTEXT USAGE ---
- STUDY_PLAN → study_plan + planner_notes
- CONCEPT_EXPLANATION → retrieved_docs
- PERFORMANCE_INSIGHT → ml_results
- QUIZ → quiz only
- GREETING / CLOSING → no extra context
- OFF_TOPIC → redirect politely

--- STEP 4: RESPONSE FORMAT ---
If SINGLE intent:
→ Give a clean, direct response

If MULTIPLE intents:
→ Use structured sections in order:
1. 📊 Performance Insight
2. 📚 Concept Explanation
3. 🗓️ Study Plan

--- RULES ---
- No raw data dumping
- No system/internal terms
- Keep it natural and helpful
- Adjust length based on complexity

--- EDGE CASES ---
- Missing data → give best general answer
- Unclear query → assume academic intent

Now generate the best possible response.
"""

    response = llm.invoke(prompt)

    return {
        "messages": [AIMessage(content=response.content)]
    }


# GRAPH CONSTRUCTION


In [ ]:
from langgraph.graph import StateGraph, END

# ------------------------------
# 1. INITIALIZE GRAPH
# ------------------------------
workflow = StateGraph(AgentState)

# ------------------------------
# 2. ADD NODES
# ------------------------------
workflow.add_node("supervisor", master_node)
workflow.add_node("analyser", analyser_node)
workflow.add_node("planner", planner_node)
workflow.add_node("retriever", retriever_node)
workflow.add_node("quizzer", quizzer_node)
workflow.add_node("end", end_node)

# ------------------------------
# 3. DEFINE FLOW
# ------------------------------

# Entry point
workflow.set_entry_point("supervisor")

# Dynamic routing from supervisor
workflow.add_conditional_edges(
    "supervisor",
    lambda state: state["next_node"],
    {
        "analyser": "analyser",
        "planner": "planner",
        "retriever": "retriever",
        "quizzer": "quizzer",
        "end": "end"
    }
)

# All nodes return to supervisor
workflow.add_edge("analyser", "supervisor")
workflow.add_edge("planner", "supervisor")
workflow.add_edge("retriever", "supervisor")
workflow.add_edge("quizzer", "supervisor")

# End node → terminate graph
workflow.add_edge("end", END)

# ------------------------------
# 4. COMPILE
# ------------------------------
app = workflow.compile()

print("✅ Graph Compiled Successfully!")


# TESTING THE AGENT


In [ ]:
# TEST 1: CONCEPT EXPLANATION


from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="Explain algebra")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],

    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "awaiting_answer": False,

    "study_plan": "",
    "planner_notes": "",

    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

In [1]:
# TEST 2: STUDY PLAN


from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="Give me a study plan to improve my math")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],

    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "awaiting_answer": False,

    "study_plan": "",
    "planner_notes": "",

    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

NameError: name 'app' is not defined

In [ ]:
# TEST 3: QUIZ SYSTEM


from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="Give me a quiz on algebra")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],

    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "awaiting_answer": False,

    "study_plan": "",
    "planner_notes": "",

    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

In [ ]:
# TEST 4: GREETING


from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(content="Hi")],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],

    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "awaiting_answer": False,

    "study_plan": "",
    "planner_notes": "",

    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

In [ ]:
# TEST 5: PERFORMANCE ANALYSIS

from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(
        content="I scored 42 in math and study only 1 hour daily. Am I performing well or not?"
    )],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],

    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "awaiting_answer": False,

    "study_plan": "",
    "planner_notes": "",

    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

In [ ]:
# TEST 6: MULTI-INTENT QUERY

from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(
        content="My math score is 48. Explain quadratic equations and give me a study plan to improve."
    )],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],

    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "awaiting_answer": False,

    "study_plan": "",
    "planner_notes": "",

    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

In [ ]:
# TEST 7: FULL SYSTEM (EXPLAIN + PLAN + QUIZ)

from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(
        content="I got around 52 in my math exam. I'm struggling with quadratic equations. Can you explain it in a simple way, suggest how I should study, and maybe give me a small quiz to practice quadratic equations?"
    )],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],

    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "awaiting_answer": False,

    "study_plan": "",
    "planner_notes": "",

    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)

In [ ]:
# TEST 8: PERFORMANCE + IMPROVEMENT

from langchain_core.messages import HumanMessage

test_input = {
    "messages": [HumanMessage(
        content="I scored 52 in math. How am I doing and what should I do to improve?"
    )],

    "student_data": {},
    "ml_results": {},
    "retrieved_docs": [],

    "quiz_questions": [],
    "current_q_idx": 0,
    "quiz_score": 0,
    "quiz_active": False,
    "awaiting_answer": False,

    "study_plan": "",
    "planner_notes": "",

    "plan": [],
    "current_step_index": 0,
    "next_node": ""
}

result = app.invoke(test_input)

print("\n--- FINAL OUTPUT ---\n")
for msg in result["messages"]:
    print(msg.content)